In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# LightGBM model:
### Packages Importing, downloading data from yahoo finances for example stock:

In [ ]:
# מייבאים את המודול yfinance להורדת נתוני מניות
import yfinance as yf
# מייבאים את הפונקציה לחלק את הנתונים לאימונים ולבדיקות
from sklearn.model_selection import train_test_split
# מייבאים את מודל LGBM לרגרסיה
from lightgbm import LGBMRegressor
# מייבאים את matplotlib לצורך יצירת גרפים
import matplotlib.pyplot as plt
# מייבאים את StandardScaler לנרמול הנתונים
from sklearn.preprocessing import StandardScaler
# מייבאים את pandas לצורך ניהול מסגרות נתונים
import pandas as pd
# מייבאים את numpy לצורך עבודה עם מערכים
import numpy as np
# מייבאים את matplotlib.dates לצורך עיצוב תאריכים בגרפים
import matplotlib.dates as mdates
# כל השגיאות ואחוז הדיוק במודל רגרסיה R^2
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# # רשימת הסימולים של המניות (תוקנה, הוסר מחרוזת ריקה)
df_tickers = pd.read_csv("/content/drive/Shareddrives/capstone project-stock market robo-advisor/symbols_Israel.csv")
tickers = df_tickers["0"].tolist()[260:]

# # הורדת נתוני המניות בין התאריכים 01-01-2016 עד 01-09-2024
data = yf.download(tickers, start="2020-01-01", end="2024-12-31", auto_adjust=False)['Adj Close']
data

### Feature Extraction:

In [ ]:
# פונקציה לציור גרף המציג את המחירים בפועל מול המחירים החזויים
def plot_actual_vs_predicted(data, y_test, pred, model_name='Model'):
    plt.figure(figsize=(16, 8))  # הגדרת גודל הגרף

    # ודא שהאינדקס של הנתונים הוא בפורמט תאריך
    data.index = pd.to_datetime(data.index)

    # ציור המחירים בפועל (בכחול) והמחירים החזויים (באדום)
    plt.plot(data.index[-len(y_test):], y_test, label='Actual', color='blue', linewidth=2)
    plt.plot(data.index[-len(y_test):], pred, label='Predicted', color='red', linewidth=2)
    plt.fill_between(data.index[-len(y_test):], y_test, pred, color='gray', alpha=0.3)  # אזור הצללה בין תחזית למחירים בפועל

    plt.xlabel('Date', fontsize=14)  # תווית ציר X
    plt.ylabel('Price', fontsize=14)  # תווית ציר Y
    plt.title(f'{model_name}: Actual vs Predicted Prices Over Time', fontsize=16)  # כותרת הגרף
    plt.legend(fontsize=12)  # הוספת אגדה

    # עיצוב ציר ה-X לתצוגת תאריכים
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

    plt.gcf().autofmt_xdate()  # סידור אוטומטי של התאריכים
    plt.tight_layout()  # התאמת הגרף לחלון התצוגה
    plt.grid(True, linestyle='--', alpha=0.7)  # הוספת רשת
    plt.show()

# מעדכנים את רשימת המניות לאחר הורדת מניות עם פחות מ 100 ימים בדאטא
tickers = data.columns.tolist()
# יצירת ממוצעים נעים (Rolling windows) עם טיפול בנתוני NaN
for window in [2, 5, 10, 20, 60]:  # עבור חלונות של 2, 5, 10, 20 ו-60 ימים
    for ticker in tickers:  # עבור כל סמל במניות
        # ממוצע נע
        data[f'{ticker}_rolling_{window}'] = data[ticker].rolling(window=window).mean().fillna(method='ffill').fillna(method='bfill')
        # סטיית תקן נעה
        data[f'{ticker}_rollingSTD_{window}'] = data[ticker].rolling(window=window).std().fillna(method='ffill').fillna(method='bfill')
        # חציון נע
        data[f'{ticker}_rollingMedian_{window}'] = data[ticker].rolling(window=window).median().fillna(method='ffill').fillna(method='bfill')

# # הוספת ממוצע נע אקספוננציאלי עבור החלונות הבאים
# for window in [20, 50, 100, 200]:
#     for ticker in tickers:
#         data[f'{ticker}_ema_{window}'] = data[ticker].ewm(span=window, adjust=False).mean()


# מילוי ערכי NaN עם הערך הקודם או הבא
data.fillna(method='bfill', inplace=True)
data.fillna(method='ffill', inplace=True)


In [ ]:
data.shape

In [ ]:
!pip install lightgbm

## 🔦 Stock Forecasting using LightGBM: Performance Evaluation + 30-Day Forecast

This notebook performs **stock-level regression forecasting** using the `LightGBM` model. It is designed to process one or more TASE stocks based on historical values of **other stocks as features**, and returns both **model performance metrics** and **30-day future forecasts**.

---

### 🎯 Objective

For each selected stock:
- Use other stocks’ historical prices as predictors (X).
- Train a LightGBM regressor to predict its price (Y).
- Evaluate model performance using:
  - R² (coefficient of determination)
  - MAE (Mean Absolute Error)
  - MSE (Mean Squared Error)
- Predict the next 30 days' values (based on last 30 feature rows).

---

### 📁 Input

- `data`: A DataFrame where each column is a stock (e.g., `"AVIV.TA"`, `"TEVA.TA"`).
  - **Target stock** is one of the columns (selected manually).
  - All **other columns** are treated as features.

---

### ⚙️ Pipeline Overview

1. **Feature Scaling**:
   - Standardizes the feature matrix (excluding the target column).

2. **Train-Test Split**:
   - **Chronological** split (no shuffle) → preserves time ordering.
   - 90% train, 10% test.

3. **Model Training**:
   - `LGBMRegressor` with:
     - `n_estimators=100`
     - `learning_rate=0.05`
     - `max_depth=6`
     - `random_state=42`

4. **Evaluation**:
   - Predictions are made on the **test set**.
   - Metrics logged:
     - **R²**: Proportion of variance explained.
     - **MAE**: Average absolute prediction error.
     - **MSE**: Average squared error.

5. **Forecasting**:
   - The model forecasts are generated by computing daily percent change statistics (mean and std) from test predictions.
   - It samples new daily returns from a normal distribution and applies them iteratively over 30 future days, starting from the last predicted price.

6. **Output Format**:
   - A single row containing:
     - `Stock` name
     - R², MAE, MSE
     - 30-day forecasts (`Day_1`, ..., `Day_30`)
     - Full test labels & predictions (`y_test`, `y_pred` as JSON strings)

---

### ✅ Sample Output (Row)

| Stock    | R-squared | MAE    | MSE     | Day_1 | Day_2 | ... | Day_30 | y_test (JSON) | y_pred (JSON) |
|----------|-----------|--------|---------|-------|--------|-----|--------|---------------|----------------|
| AVIV.TA  | 0.84      | 2.13   | 5.67    | 542.1 | 543.2  | ... | 553.0  | [...]         | [...]          |





In [ ]:
from re import L
import os
import json
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
import pickle

# ignore warnings:
import warnings
warnings.filterwarnings('ignore')
# Define the output CSV file
output_csv_path = "LightGBM_metrics_and_predictions.csv"

# Get the list of stocks

###### Change here the number of stocks ######
stocks = data.columns

# Predefine the columns
base_columns = ['Stock', 'R-squared', 'MAE', 'MSE', 'MAPE'] + [f"Day_{i+1}" for i in range(30)] + ["y_test", "y_pred"]

# Initialize the empty metrics DataFrame
metrics_df = pd.DataFrame(columns=base_columns)

# # Check if the output file exists
# if os.path.exists(output_csv_path):
#     # Load the existing CSV to find processed stocks
#     existing_df = pd.read_csv(output_csv_path)
#     processed_stocks = set(existing_df["Stock"].unique())
#     stock_number_offset = len(existing_df)  # Start numbering from the existing number
# else:
#     # No file exists, start fresh
#     processed_stocks = set()
#     stock_number_offset = 0

# # Loop through all stocks
# num_skip = 0
# for i, ticker in enumerate(stocks, start=1):
#     # Skip stocks that are already processed
#     if ticker in processed_stocks:
#         print(f"Skipping already processed stock: {ticker}")
#         num_skip += 1
#         continue

for ticker in tickers:

    print(f"Processing stock: {ticker}")

    try:
        # Normalize independent variables (X), excluding the target stock
        scaler_X = StandardScaler()
        X_scaled = pd.DataFrame(scaler_X.fit_transform(data.drop(columns=[ticker])),
                                columns=data.columns.drop(ticker),
                                index=data.index)

        # Define the target variable
        y = data[ticker]

        # Split data into training and testing sets (chronological order)
        X_train, X_test, y_train, y_test, train_dates, test_dates = train_test_split(
            X_scaled, y, X_scaled.index, test_size=0.1, random_state=42, shuffle=False)

        # Train the LightGBM model
        model = LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42)
        model.fit(X_train, y_train)

        # Predict on test data
        y_pred = model.predict(X_test)

        # Calculate Metrics
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)

        # Prepare data for the last 30 days
        X_last_30 = X_scaled.iloc[-30:].copy()
        X_scaled = X_scaled.iloc[:-30]

        # Predict future values for the next 30 days

        # Simulate forecast path from last predicted price
        future_predictions = [y_pred[-1]]  # Start from last known prediction

        for _ in range(30):
          # Compute percent changes from test predictions & future forecasts:
            pct_changes = pd.Series(y_pred).pct_change().dropna()
            mean_pct_change = pct_changes.mean()
            std_pct_change = pct_changes.std()
            sampled_pct_change = np.random.normal(loc=mean_pct_change, scale=std_pct_change)
            new_forecast = future_predictions[-1] * (1 + sampled_pct_change)
            future_predictions.append(new_forecast)
            y_pred.append(new_forecast)
        y_pred=y_pred[:-30]
        # Strip the first value (initial known point) to get only new forecasts
        y_future_pred = future_predictions[1:]

        # Prepare columns for future predictions (Day_1, Day_2, ..., Day_30)
        prediction_columns = {f"Day_{i+1}": y_future_pred[i] for i in range(len(y_future_pred))}

        # Create a single row of data for the stock
        metrics_data = {
            # "Stock_Number": stock_number_offset + i - num_skip,  # Add stock number
            "Stock": ticker,
            "R-squared": r2,
            "MAE": mae,
            "MSE": mse,
            "MAPE": mape,
            **prediction_columns,  # Add the dynamic prediction columns
            "y_test": json.dumps(y_test.tolist()),  # Serialize y_test as a JSON string
            "y_pred": json.dumps(y_pred.tolist())   # Serialize y_pred as a JSON string
        }

        metrics_df_temp = pd.DataFrame([metrics_data])
        metrics_df = pd.concat([metrics_df, metrics_df_temp], ignore_index=True)

        # Define path
        os.makedirs("/content/drive/Shareddrives/capstone project-stock market robo-advisor/LightGBM models", exist_ok=True)

        model_path = f"/content/drive/Shareddrives/capstone project-stock market robo-advisor/LightGBM models/{ticker.replace('.', '_')}_LightGBM_model.pkl"

        # Save LightGBM model with pickle
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)

        print(f"✅ LightGBM model for {ticker} saved to: {model_path}")

        # Calculate standard deviation of predictions
        std_pred = np.std(y_pred)

        # Print all metrics
        print(f"Metrics for stock: {ticker}")
        print(f"R-squared (R²): {r2:.4f}")
        print(f"Mean Absolute Error (MAE): {mae:.4f}")
        print(f"Mean Absolute Percentage Error (MAE): {mape:.4f}")
        print(f"Mean Squared Error (MSE): {mse:.4f}")
        print(f"Standard Deviation of Predictions: {std_pred:.4f}")

        # Create a timeline for plotting
        # - y_test and y_pred follow the test_dates
        # - y_future_pred follows the 30 days after the last date

        test_dates = pd.Series(test_dates)

        future_dates = pd.date_range(start=test_dates.iloc[-1] + pd.Timedelta(days=1), periods=30, freq='D')

        # Plot
        plt.figure(figsize=(14, 8))

        # Plot actual test values
        plt.plot(test_dates, y_test.values, label='Actual (y_test)', marker='o')

        # Plot predicted test values
        plt.plot(test_dates, y_pred, label='Predicted (y_pred)', marker='x')

        # Plot future forecast
        plt.plot(future_dates, y_future_pred, label='Forecast (next 30 days)', marker='^')

        plt.title(f"Stock: {ticker} — Actual vs Predicted vs Forecasted")
        plt.xlabel('Date')
        plt.ylabel('Stock Price')
        plt.legend()
        plt.grid(True)

        # Annotate metrics on the plot
        metric_text = (f"R²: {r2:.4f}\n"
                      f"MAE: {mae:.4f}\n"
                      f"MAPE: {mape:.4f}\n"
                      f"MSE: {mse:.4f}\n"
                      f"STD of Predictions: {std_pred:.4f}")
        plt.gcf().text(0.15, 0.75, metric_text, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))

        plt.tight_layout()
        plt.show()

        # # Append to CSV if it exists, otherwise create a new file
        # if os.path.exists(output_csv_path):
        #     existing_df = pd.read_csv(output_csv_path)
        #     updated_df = pd.concat([existing_df, metrics_df], ignore_index=True)
        #     updated_df.to_csv(output_csv_path, index=False)
        # else:
        #     metrics_df.to_csv(output_csv_path, index=False)

        # print(f"Metrics and future predictions for {ticker} saved to {output_csv_path}")

    except Exception as e:
        # Log the error and continue with the next stock
        print(f"An error occurred while processing stock {ticker}: {e}")
        # num_skip += 1
